In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/nasa_feature.csv")

print(df.head())
print(df.columns)

   Cycle  Capacity       Rct        Re  ...    tau_p1    tau_p2  tau_p3    tau_p4
0      1  1.856487  0.069456  0.044669  ...  1.856487  0.128945     1.0  0.489269
1      2  1.846327  0.076275  0.046687  ...  1.846327  0.140828     1.0  0.509916
2      3  1.835349  0.067972  0.044843  ...  1.835349  0.124752     1.0  0.478502
3      4  1.835263  0.074534  0.046195  ...  1.835263  0.136789     1.0  0.501043
4      5  1.834646  0.068528  0.045101  ...  1.834646  0.125725     1.0  0.480272

[5 rows x 14 columns]
Index(['Cycle', 'Capacity', 'Rct', 'Re', 'SOH', 'tau_real', 'CCA1', 'CCA2',
       'CCA3', 'CCA4', 'tau_p1', 'tau_p2', 'tau_p3', 'tau_p4'],
      dtype='object')


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io

# 데이터 로드
mat = scipy.io.loadmat("../data/NASA/5. Battery Data Set/1. BatteryAgingARC-FY08Q4/B0005.mat")  # 네 파일명 맞게 수정
cycle = mat["B0005"][0,0]["cycle"][0]

print(len(cycle))

616


In [4]:
print(cycle[0].dtype)

[('type', 'O'), ('ambient_temperature', 'O'), ('time', 'O'), ('data', 'O')]


In [5]:
print(cycle[0]["ambient_temperature"])

[[24]]


In [6]:
temperatures = []
for record in cycle:
    if record["type"][0] == "discharge":
        temp = record["ambient_temperature"][0][0]
        temperatures.append(temp) 


In [7]:
df["Temperature"] = temperatures

In [8]:
df.head()

,Cycle,Capacity,Rct,Re,SOH,tau_real,CCA1,CCA2,CCA3,CCA4,tau_p1,tau_p2,tau_p3,tau_p4,Temperature
0,1,1.856487,0.069456,0.044669,1.000000,0.128945,26.728866,1.856487,14.397548,7.044275,1.856487,0.128945,1.0,0.489269,24
1,2,1.846327,0.076275,0.046687,0.994527,0.140828,24.206274,1.846327,13.110500,6.685260,1.846327,0.140828,1.0,0.509916,24
2,3,1.835349,0.067972,0.044843,0.988614,0.124752,27.001525,1.835349,14.711928,7.039689,1.835349,0.124752,1.0,0.478502,24
3,4,1.835263,0.074534,0.046195,0.988567,0.136789,24.623196,1.835263,13.416716,6.722353,1.835263,0.136789,1.0,0.501043,24
4,5,1.834646,0.068528,0.045101,0.988235,0.125725,26.772074,1.834646,14.592505,7.008371,1.834646,0.125725,1.0,0.480272,24


In [9]:
print(df["Temperature"].value_counts())

Temperature
24    168
Name: count, dtype: int64


In [10]:
df[["Temperature","SOH"]].corr()

,Temperature,SOH
Temperature,NaN,NaN
SOH,NaN,1.0


# NASA ambient_temperature 분석
- NASA 데이터의 ambient_temperature는 일정하게 유지되며 변동성이 없다.
따라서 온도 변수는 해당 데이터셋에서는 유의미한 설명 변수로 작용하지 않았다.

In [11]:
df_temp = pd.read_excel("../data/processed/_Work_temperature.xlsx")

In [12]:
df_temp["날짜"] = pd.to_datetime(df_temp["날짜"])

In [13]:
print(df_temp.columns)

Index(['날짜', '차량', '배터리 종류', '전압(V)', '내부저항(mΩ)', 'SOH(%)', 'CCA(A)',
       'RC(min)', '교체 여부', '메모', '증상', '테스트 결과', '기온'],
      dtype='object')


In [14]:
df_temp["tau_p4"] = df_temp["CCA(A)"] * (df_temp["내부저항(mΩ)"] ** 0.5)

In [15]:
df_temp[["CCA(A)", "내부저항(mΩ)", "tau_p4","기온"]].head()

,CCA(A),내부저항(mΩ),tau_p4,기온
0,NaN,NaN,NaN,30.0
1,459.0,5.89,1113.961889,32.5
2,NaN,NaN,NaN,31.8
3,NaN,NaN,NaN,28.5
4,103.0,26.11,526.308835,31.9


In [16]:
df_temp = df_temp.dropna(subset=["tau_p4"])

In [17]:
import joblib

model = joblib.load("model.pkl")

c:\Users\Administrator\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [18]:
df_temp["SOH_pred"] = model.predict(df_temp[["tau_p4"]])

In [19]:
# 기존 모델 보정
from sklearn.linear_model import LinearRegression

cal_model = LinearRegression()
cal_model.fit(df_temp[["SOH_pred"]], df_temp["SOH(%)"])

df_temp["SOH_final"] = cal_model.predict(df_temp[["SOH_pred"]])

In [20]:
from sklearn.metrics import r2_score

r2_old = r2_score(df_temp["SOH(%)"], df_temp["SOH_final"])
print("기존 모델 R2:", r2_old)

기존 모델 R2: 0.8072413230341609


In [21]:
# 온도 포함 모델
X = df_temp[["tau_p4", "기온"]]
y = df_temp["SOH(%)"]

model_temp = LinearRegression()
model_temp.fit(X, y)

LinearRegression()